# Task 4: Custom Gradient Descent Optimizers with Momentum, RMSprop, and Adam

## Objective

Implement Momentum, RMSprop, and Adam optimizers from scratch using PyTorch tensor mathematics.

The experiment compares the convergence behavior of the optimizers on a non-linear classification dataset.

The update rules are implemented manually without using `torch.optim`.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# ------------------------------------------------------------
# Non-linear two-moons dataset
# ------------------------------------------------------------

N = 1200

t = np.linspace(0, np.pi, N // 2)

x1 = np.c_[np.cos(t), np.sin(t)]
x2 = np.c_[1 - np.cos(t), 1 - np.sin(t) - 0.5]

X = np.vstack([x1, x2])
y = np.hstack([
    np.zeros(N // 2),
    np.ones(N // 2)
])

# Add noise
X += np.random.normal(0, 0.12, X.shape)

# Shuffle
idx = np.random.permutation(N)

X = torch.tensor(
    X[idx],
    dtype=torch.float32
)

y = torch.tensor(
    y[idx],
    dtype=torch.float32
).reshape(-1, 1)

# Train/test split
split = int(0.8 * N)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

plt.figure(figsize=(7, 5))
plt.scatter(
    X[:, 0],
    X[:, 1],
    c=y.squeeze(),
    alpha=0.7
)
plt.title("Non-Linear Two-Moons Dataset")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

# Neural Network and Manual Optimizer State

A small non-linear neural network is used:

\[
2 \rightarrow 32 \rightarrow 32 \rightarrow 1
\]

ReLU is used in the hidden layers and Sigmoid in the output layer.

The model parameters are PyTorch tensors with gradients enabled. The actual parameter updates are performed manually using tensor mathematics.

In [ ]:
# ============================================================
# Neural Network
# ============================================================

def create_model():

    # Create tensors first, then enable gradients.
    # This keeps them as leaf tensors.
    W1 = (torch.randn(2, 32) * 0.1).requires_grad_()
    b1 = torch.zeros(32, requires_grad=True)

    W2 = (torch.randn(32, 32) * 0.1).requires_grad_()
    b2 = torch.zeros(32, requires_grad=True)

    W3 = (torch.randn(32, 1) * 0.1).requires_grad_()
    b3 = torch.zeros(1, requires_grad=True)

    return [
        W1, b1,
        W2, b2,
        W3, b3
    ]


def forward(X, params):

    W1, b1, W2, b2, W3, b3 = params

    h1 = torch.relu(X @ W1 + b1)
    h2 = torch.relu(h1 @ W2 + b2)

    output = torch.sigmoid(
        h2 @ W3 + b3
    )

    return output


def loss_fn(pred, target):

    eps = 1e-7

    pred = torch.clamp(
        pred,
        eps,
        1 - eps
    )

    return -torch.mean(
        target * torch.log(pred)
        + (1 - target) * torch.log(1 - pred)
    )


# Check that every parameter is a leaf tensor
params_test = create_model()

for i, p in enumerate(params_test):
    print(
        f"Parameter {i+1}: "
        f"leaf={p.is_leaf}, "
        f"requires_grad={p.requires_grad}"
    )

# Custom Optimizers

All optimizer update rules are implemented from scratch.

No `torch.optim` is used.

Momentum stores an exponentially weighted first moment.

RMSprop stores an exponentially weighted second raw moment.

Adam stores both moments and applies bias correction before updating the parameters.

In [ ]:
# ============================================================
# Custom Optimizer
# ============================================================

class CustomOptimizer:

    def __init__(
        self,
        params,
        name,
        lr=0.01,
        beta1=0.9,
        beta2=0.999,
        eps=1e-8
    ):

        self.params = params
        self.name = name
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0

        # Optimizer states
        self.m = [
            torch.zeros_like(p)
            for p in params
        ]

        self.v = [
            torch.zeros_like(p)
            for p in params
        ]

    def step(self):

        self.t += 1

        with torch.no_grad():

            for i, p in enumerate(self.params):

                g = p.grad

                if self.name == "momentum":

                    self.m[i] = (
                        self.beta1 * self.m[i]
                        + (1 - self.beta1) * g
                    )

                    p -= self.lr * self.m[i]

                elif self.name == "rmsprop":

                    self.v[i] = (
                        self.beta2 * self.v[i]
                        + (1 - self.beta2) * g**2
                    )

                    p -= (
                        self.lr * g
                        / (
                            torch.sqrt(self.v[i])
                            + self.eps
                        )
                    )

                elif self.name == "adam":

                    self.m[i] = (
                        self.beta1 * self.m[i]
                        + (1 - self.beta1) * g
                    )

                    self.v[i] = (
                        self.beta2 * self.v[i]
                        + (1 - self.beta2) * g**2
                    )

                    # Bias correction
                    m_hat = (
                        self.m[i]
                        / (1 - self.beta1**self.t)
                    )

                    v_hat = (
                        self.v[i]
                        / (1 - self.beta2**self.t)
                    )

                    p -= (
                        self.lr
                        * m_hat
                        / (
                            torch.sqrt(v_hat)
                            + self.eps
                        )
                    )

                else:
                    raise ValueError(
                        "Unknown optimizer"
                    )

                p.grad.zero_()

# Training Harness

Each optimizer is trained independently from the same initial model.

The loss is recorded at every epoch so that the convergence rates can be compared.

In [ ]:
# ============================================================
# Training
# ============================================================

def train(optimizer_name, epochs=150):

    params = create_model()

    optimizer = CustomOptimizer(
        params,
        optimizer_name,
        lr=0.01
    )

    losses = []
    accuracies = []

    for epoch in range(epochs):

        # Forward
        predictions = forward(
            X_train,
            params
        )

        loss = loss_fn(
            predictions,
            y_train
        )

        # Backward using PyTorch autograd
        loss.backward()

        # Manual optimizer update
        optimizer.step()

        # Evaluation
        with torch.no_grad():

            test_pred = forward(
                X_test,
                params
            )

            test_loss = loss_fn(
                test_pred,
                y_test
            )

            accuracy = (
                (test_pred >= 0.5)
                == y_test
            ).float().mean()

        losses.append(
            loss.item()
        )

        accuracies.append(
            accuracy.item()
        )

        if (epoch + 1) % 30 == 0:
            print(
                f"{optimizer_name.upper():8s} | "
                f"Epoch {epoch+1:3d} | "
                f"Loss: {loss.item():.4f} | "
                f"Accuracy: {accuracy.item()*100:.2f}%"
            )

    return losses, accuracies


results = {}

for optimizer_name in [
    "momentum",
    "rmsprop",
    "adam"
]:

    results[optimizer_name] = train(
        optimizer_name
    )

In [ ]:
# ============================================================
# Convergence comparison
# ============================================================

plt.figure(figsize=(9, 5))

for name, (losses, _) in results.items():

    plt.plot(
        losses,
        label=name.upper()
    )

plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Loss Convergence Comparison")
plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# Accuracy comparison
# ============================================================

plt.figure(figsize=(9, 5))

for name, (_, accuracies) in results.items():

    plt.plot(
        np.array(accuracies) * 100,
        label=name.upper()
    )

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Comparison")
plt.legend()
plt.grid(True)
plt.show()


# Final results
print("========== FINAL RESULTS ==========")

for name, (losses, accuracies) in results.items():

    print(
        f"{name.upper():8s} | "
        f"Final Loss: {losses[-1]:.4f} | "
        f"Final Accuracy: {accuracies[-1]*100:.2f}%"
    )

# Conclusion

Momentum, RMSprop, and Adam optimizers were implemented from scratch using PyTorch tensor mathematics without using `torch.optim`.

Momentum introduced a first-moment moving average to smooth gradient updates.

RMSprop used an exponentially weighted second raw moment to adapt the learning rate for each parameter.

Adam combined both first and second moments and applied bias correction:

\[
\hat m_t=\frac{m_t}{1-\beta_1^t},
\qquad
\hat v_t=\frac{v_t}{1-\beta_2^t}
\]

The loss and accuracy curves demonstrate the different convergence behaviors of the optimizers on a non-linear classification problem.

This experiment demonstrates how temporal gradient information and adaptive learning rates can improve optimization compared with basic gradient descent.